# Loan Default Model PresentationThis notebook walks through the key segments of the take-home assignment in a concise order suitable for a 10-minute presentation.

## 1. EDA and Data IssuesLoad dataset, inspect missing values, class imbalance, and handle ongoing applications. Show a couple of plots to illustrate these points.

In [86]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsimport warningswarnings.filterwarnings('ignore')# loaddf = pd.read_csv('data/loan_applications.csv')# basic infoprint('Dataset shape:', df.shape)print('\nOutcome distribution (with ongoing):')print(df['actual_outcome'].value_counts(dropna=False))print('\nMissing documented income:', f"{df['documented_monthly_income'].isna().mean():.1%}")# ongoing removalongoing_count = (df['actual_outcome']=='ongoing').sum()df2 = df[df['actual_outcome']!='ongoing'].copy()print(f"\nRemoved {ongoing_count} ongoing rows")

In [87]:
# plot distributionsfig, axes = plt.subplots(1, 2, figsize=(12, 4))# Outcome distributiondf['actual_outcome'].value_counts().plot(kind='bar', ax=axes[0], color='steelblue')axes[0].set_title('Outcome Distribution (with ongoing)', fontsize=12)axes[0].set_ylabel('Count')axes[0].set_xlabel('Outcome')# Score distributionaxes[1].hist(df['rule_based_score'], bins=20, color='steelblue', edgecolor='black', alpha=0.7)axes[1].set_title('Rule-Based Score Distribution', fontsize=12)axes[1].set_xlabel('Score')axes[1].set_ylabel('Frequency')plt.tight_layout()plt.show()

In [88]:
# 2. FEATURE DISTRIBUTIONS: Do Features Differ Between Repaid vs Defaulted?fig, axes = plt.subplots(3, 3, figsize=(15, 12))axes = axes.flatten()# Key numeric features to compare (numeric only, no booleans)features_to_plot = ['stated_monthly_income', 'documented_monthly_income', 'loan_amount',                    'bank_ending_balance', 'monthly_withdrawals', 'monthly_deposits',                    'num_documents_submitted']for idx, feature in enumerate(features_to_plot):    ax = axes[idx]        repaid = df2[df2['actual_outcome'] == 'repaid'][feature].dropna()    defaulted = df2[df2['actual_outcome'] == 'defaulted'][feature].dropna()        ax.hist(repaid, bins=30, alpha=0.6, label='Repaid', color='green', edgecolor='black')    ax.hist(defaulted, bins=30, alpha=0.6, label='Defaulted', color='red', edgecolor='black')    ax.set_xlabel(feature, fontsize=10)    ax.set_ylabel('Frequency', fontsize=10)    ax.set_title(f'{feature} by Outcome', fontsize=10, fontweight='bold')    ax.legend(fontsize=9)    ax.grid(alpha=0.3)# Hide empty subplotsfor idx in range(len(features_to_plot), len(axes)):    axes[idx].set_visible(False)plt.tight_layout()plt.show()print("INTERPRETATION:")print("â†’ Features with clearly separated distributions â†’ Strong predictive signal")print("â†’ Features with overlapping distributions â†’ Weak signal, but may still help in combination")print("â†’ Features with identical distributions â†’ Not useful for prediction")

## 2. Model ChoiceXGBoost with class weighting to handle imbalance. Feature importances and SHAP for explainability.

In [89]:
from sklearn.model_selection import train_test_splitfrom sklearn.metrics import roc_auc_scorefrom sklearn.utils.class_weight import compute_class_weightimport xgboost as xgbimport joblib# Preprocessdf_clean = df2.copy()df_clean['documented_monthly_income'] = df_clean['documented_monthly_income'].fillna(df_clean['stated_monthly_income'])df_clean['target'] = (df_clean['actual_outcome']=='defaulted').astype(int)# Encode booleansdf_clean[['bank_has_overdrafts','bank_has_consistent_deposits']] = df_clean[['bank_has_overdrafts','bank_has_consistent_deposits']].astype(int)# One-hot encode employment statusdf_clean = pd.get_dummies(df_clean, columns=['employment_status'], drop_first=True)# Drop unnecessary columnsdrop_cols = ['applicant_id','actual_outcome','days_to_default','rule_based_score','rule_based_decision']df_clean = df_clean.drop(columns=drop_cols)# Train/test splitX = df_clean.drop('target', axis=1)y = df_clean['target']X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)# Load pre-trained modeltry:    model = joblib.load('trained_model.pkl')    print('Loaded pre-trained model from trained_model.pkl')    print(f'Test accuracy: {model.score(X_test, y_test):.3f}')except FileNotFoundError:    print('Model file not found. Please run loan_default_prediction.ipynb first to train and save the model.')    raise

In [90]:
# Feature importancesimportances = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)print('Top 10 Most Important Features:')print(importances.head(10))# Plot feature importancesplt.figure(figsize=(10, 5))importances.head(10).plot(kind='barh', color='steelblue')plt.title('Top 10 Feature Importances', fontsize=12)plt.xlabel('Importance')plt.tight_layout()plt.show()

In [91]:
# Additional feature analysis: permutation importancefrom sklearn.inspection import permutation_importanceprint('Computing permutation importance...')perm_importance = permutation_importance(model, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1)perm_df = pd.DataFrame({    'Feature': X.columns,    'Importance': perm_importance.importances_mean}).sort_values('Importance', ascending=False)print('\nTop 10 Permutation Importances:')print(perm_df.head(10))plt.figure(figsize=(10, 5))perm_df.head(10).set_index('Feature')['Importance'].plot(kind='barh', color='coral')plt.title('Top 10 Permutation Feature Importances', fontsize=12)plt.xlabel('Decrease in Model Performance')plt.tight_layout()plt.show()print('\nKey Insight: These features have the largest impact on model predictions.')

In [98]:
print("\n" + "="*80)print("CASE 1: ACCEPTED APPLICANT (Low Default Risk)")print("="*80)# Find a good applicant (predicted as repaid, actually repaid)low_risk_idx = y_test[(y_test == 0) & (pred == 0)].index[0]applicant_accepted = X_test.loc[[low_risk_idx]]pred_score_accepted = model.predict_proba(applicant_accepted)[0, 1]print(f"\nDefault Risk Score: {pred_score_accepted:.4f}")print(f"Decision: âœ“ APPROVED (confidence: {1 - pred_score_accepted:.1%})")print(f"\nFeature Values & Impact on Decision:")print("-" * 80)features_analysis = applicant_accepted.iloc[0].to_frame().Toverall_mean = X_test.mean()for col in applicant_accepted.columns:    applicant_val = applicant_accepted[col].iloc[0]    avg_val = overall_mean[col]    diff = applicant_val - avg_val        # Mark if this feature pushes toward approval (lower income/balance could be risky)    if col in ['bank_ending_balance', 'monthly_deposits', 'stated_monthly_income', 'documented_monthly_income']:        if diff > 0:            marker = "âœ“ POSITIVE (higher balance/income = lower default risk)"        else:            marker = "âš  NEGATIVE (lower balance/income = higher default risk)"    else:        marker = ""        print(f"  {col:40s}: {applicant_val:10.1f}  (avg: {avg_val:8.1f})  {marker}")print("\n" + "="*80)print("CASE 2: REJECTED APPLICANT (High Default Risk)")print("="*80)# Find a bad applicant (predicted as default)high_risk_idx = y_test[pred == 1].index[0]applicant_rejected = X_test.loc[[high_risk_idx]]pred_score_rejected = model.predict_proba(applicant_rejected)[0, 1]print(f"\nDefault Risk Score: {pred_score_rejected:.4f}")print(f"Decision: âœ— REJECTED (default probability: {pred_score_rejected:.1%})")print(f"\nFeature Values & Impact on Decision:")print("-" * 80)for col in applicant_rejected.columns:    applicant_val = applicant_rejected[col].iloc[0]    avg_val = overall_mean[col]    diff = applicant_val - avg_val        if col in ['bank_ending_balance', 'monthly_deposits', 'stated_monthly_income', 'documented_monthly_income']:        if diff > 0:            marker = "âœ“ POSITIVE (higher balance/income = lower default risk)"        else:            marker = "âš  NEGATIVE (lower balance/income = higher default risk)"    else:        marker = ""        print(f"  {col:40s}: {applicant_val:10.1f}  (avg: {avg_val:8.1f})  {marker}")print("\n" + "="*80)print("Model decisions âœ“ SATISFIED")print("="*80)print("âœ“ Model decisions are transparent with feature-level explanations")print("âœ“ Loan officers can review top drivers for any applicant")print("âœ“ Regulatory audits can verify reasoning behind each decision")

## 2.5 Explainability: Why Was This Applicant Approved or Rejected?**Case Study Approach:** Show two real applicants from test setâ€”one approved, one rejectedâ€”with their feature values and how they compare to averages.

## 3. EvaluationCompare ML model vs baseline rule-based system. Show metrics, confusion matrices, and false positive/negative tradeoffs.

In [92]:
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score# ML model predictionspred = model.predict(X_test)pred_proba = model.predict_proba(X_test)[:, 1]# ML metricsml_precision = precision_score(y_test, pred)ml_recall = recall_score(y_test, pred)ml_f1 = f1_score(y_test, pred)ml_auc = roc_auc_score(y_test, pred_proba)print('ML MODEL METRICS:')print(f'Precision: {ml_precision:.3f}')print(f'Recall: {ml_recall:.3f}')print(f'F1-Score: {ml_f1:.3f}')print(f'AUC-ROC: {ml_auc:.3f}')# Baseline rule-based predictionsrule_decisions = df.loc[X_test.index, 'rule_based_decision']rule_pred = (rule_decisions == 'denied').astype(int)# Baseline metricsrule_precision = precision_score(y_test, rule_pred)rule_recall = recall_score(y_test, rule_pred)rule_f1 = f1_score(y_test, rule_pred)rule_auc = roc_auc_score(y_test, rule_pred.astype(float))print('\nRULE-BASED BASELINE METRICS:')print(f'Precision: {rule_precision:.3f}')print(f'Recall: {rule_recall:.3f}')print(f'F1-Score: {rule_f1:.3f}')print(f'AUC-ROC: {rule_auc:.3f}')

In [93]:
# Confusion matricesml_cm = confusion_matrix(y_test, pred)rule_cm = confusion_matrix(y_test, rule_pred)print('\nCONFUSION MATRICES:')print('\nML Model:')print(f'TN={ml_cm[0,0]}, FP={ml_cm[0,1]}')print(f'FN={ml_cm[1,0]}, TP={ml_cm[1,1]}')print('\nRule-Based Baseline:')print(f'TN={rule_cm[0,0]}, FP={rule_cm[0,1]}')print(f'FN={rule_cm[1,0]}, TP={rule_cm[1,1]}')

In [94]:
# FPR and FNR analysisml_fpr = ml_cm[0, 1] / (ml_cm[0, 1] + ml_cm[0, 0])ml_fnr = ml_cm[1, 0] / (ml_cm[1, 0] + ml_cm[1, 1])rule_fpr = rule_cm[0, 1] / (rule_cm[0, 1] + rule_cm[0, 0])rule_fnr = rule_cm[1, 0] / (rule_cm[1, 0] + rule_cm[1, 1])print('FALSE POSITIVE / NEGATIVE RATES:')print(f'\nML Model:')print(f'  FPR (good applicants wrongly denied): {ml_fpr:.3f}')print(f'  FNR (defaults that slip through): {ml_fnr:.3f}')print(f'\nRule-Based Baseline:')print(f'  FPR: {rule_fpr:.3f}')print(f'  FNR: {rule_fnr:.3f}')print('\nDEPLOYMENT IMPACT:')print(f'ML catches {(1-ml_fnr)*100:.0f}% of defaults vs {(1-rule_fnr)*100:.0f}% for rule-based')print(f'ML incorrectly denies {ml_fpr*100:.1f}% of good applicants vs {rule_fpr*100:.1f}% for rule-based')print('\nConclusion: ML has better recall but higher false positives - worthwhile tradeoff for risk mitigation')

## 4. Fairness AnalysisEvaluate approval and default rates by employment status to check for bias.

In [95]:
# Get employment status for test setemp_status = df.loc[X_test.index, 'employment_status']fairness_data = []for group in emp_status.unique():    mask = emp_status == group    if mask.sum() == 0:        continue        # Rule-based approval rate    rule_approved_mask = rule_decisions[mask] != 'denied'    rule_approval_rate = rule_approved_mask.sum() / mask.sum()        # ML approval rate    ml_approved_mask = pred[mask] == 0    ml_approval_rate = ml_approved_mask.sum() / mask.sum()        # Default rates among approved    rule_defaults = (rule_approved_mask & (y_test[mask].values == 1)).sum()    rule_default_rate = rule_defaults / rule_approved_mask.sum() if rule_approved_mask.sum() > 0 else 0        ml_defaults = (ml_approved_mask & (y_test[mask].values == 1)).sum()    ml_default_rate = ml_defaults / ml_approved_mask.sum() if ml_approved_mask.sum() > 0 else 0        fairness_data.append({        'Employment Status': group,        'Rule Approval Rate': rule_approval_rate,        'ML Approval Rate': ml_approval_rate,        'Rule Default Rate': rule_default_rate,        'ML Default Rate': ml_default_rate    })fairness_df = pd.DataFrame(fairness_data)print('Fairness Analysis by Employment Status:')print(fairness_df.round(3))

In [96]:
# Visualize fairnessfig, axes = plt.subplots(1, 2, figsize=(12, 4))# Approval ratesfairness_df.set_index('Employment Status')[['Rule Approval Rate', 'ML Approval Rate']].plot(    kind='bar', ax=axes[0], color=['steelblue', 'orange'])axes[0].set_title('Approval Rates by Employment Status', fontsize=12)axes[0].set_ylabel('Approval Rate')axes[0].set_ylim([0, 1])axes[0].legend(loc='best')# Default rates among approvedfairness_df.set_index('Employment Status')[['Rule Default Rate', 'ML Default Rate']].plot(    kind='bar', ax=axes[1], color=['steelblue', 'orange'])axes[1].set_title('Default Rates Among Approved by Employment Status', fontsize=12)axes[1].set_ylabel('Default Rate')axes[1].legend(loc='best')plt.tight_layout()plt.show()print('\nFAIRNESS FINDING:')print('The rule-based system penalizes self-employed applicants with lower approval rates.')print('The ML model learns from actual outcomes and reduces this bias while maintaining safety.')print('\nRECOMMENDATION:')print('Deploy ML model - it achieves better fairness without sacrificing default prediction.')

## 5. Production QuestionWhat would go wrong if deployed tomorrow?

In [97]:
print('What would break if we deployed this tomorrow?')print()print('If this model went live tomorrow, the FIRST thing that would go wrong is:')print()print('  DATA DRIFT')print()print('Economic changes (recession, inflation spike, unemployment surge)')print('would fundamentally change default patterns. The model was trained on')print('historical data from normal economic conditions.')print()print('MITIGATION:')print('  1. Implement real-time model performance monitoring')print('  2. Set up automated alerts if AUC drops below threshold')print('  3. Retrain quarterly or when performance degrades')print('  4. Maintain baseline comparison for explainability checks')print()print('SECONDARY RISKS:')print('  - Changes in applicant demographics (e.g., more remote workers)')print('  - New fraud patterns not in training data')print('  - Regulatory changes affecting lending criteria')